# Solar Filament Segmentation - V26 real-data YOLO + U-Net refiner

This version fixes the `0.00` submission at its source: the previous checkpoint was trained only on synthetic random images and produced one semantic mask per image. V26 trains on the competition's real COCO annotations, predicts individual filament instances, refines each instance boundary, and uses `pycocotools` for the official compressed RLE.

**Kaggle settings:** GPU on, Internet on. The attached `best_solar_unet` dataset is not used.

Implementation adapted from the public Apache-2.0 Kaggle notebook [YOLO + U-Net | Solar Filament Segmentation](https://www.kaggle.com/code/ektarr/yolo-u-net-solar-filament-segmentation), which reported a public score of 0.69.

In [ ]:
%pip install -q "ultralytics>=8.3,<9" pycocotools

In [ ]:
# ====================================================
# Imports
# ====================================================

import os
import json
import random
import warnings

import numpy as np
import pandas as pd

from sklearn.model_selection import train_test_split
from tqdm.auto import tqdm

import torch
from ultralytics import YOLO
from pycocotools import mask as mask_utils

warnings.filterwarnings("ignore")

import ultralytics.utils.tqdm as ultra_tqdm
ultra_tqdm.is_noninteractive_console = lambda: True

In [ ]:
# ====================================================
# Config
# ====================================================

class CFG:

    ROOT = "/kaggle/input/competitions/filament-segmentation-2026/MAGFiLO_1.0_Kaggle_2026"

    TRAIN_DIR = os.path.join(ROOT, "train", "train_images")
    TEST_DIR = os.path.join(ROOT, "test", "test_images")

    TRAIN_JSON = os.path.join(
        ROOT,
        "train",
        "MAGFiLO_1.0_Annotations_kaggle2026_train.json"
    )

    YOLO_DIR = "/kaggle/working/yolo_dataset"

    DEVICE = 0 if torch.cuda.is_available() else "cpu"
    DEBUG = False

    IMGSZ = 1024
    BATCH_SIZE = 4
    EPOCHS = 1 if DEBUG else 30
    PATIENCE = 5

    MODEL = "yolo11s-seg.pt"

    USE_CATEGORIES = True

    MASK_RATIO = 2

    CONF_THRESHOLD = 0.25
    MIN_COMPONENT_SIZE = 20

    SEED = 42

In [ ]:
# Fail early if the competition input layout is not attached correctly.
required_paths = [CFG.TRAIN_DIR, CFG.TEST_DIR, CFG.TRAIN_JSON]
missing_paths = [path for path in required_paths if not os.path.exists(path)]
if missing_paths:
    raise FileNotFoundError(f'Missing competition inputs: {missing_paths}')
print({
    'train_images': len(os.listdir(CFG.TRAIN_DIR)),
    'test_images': len(os.listdir(CFG.TEST_DIR)),
    'annotations': CFG.TRAIN_JSON,
})

In [ ]:
# ====================================================
# Seed
# ====================================================

def seed_everything(seed):
    random.seed(seed)
    np.random.seed(seed)
    torch.manual_seed(seed)
    if torch.cuda.is_available():
        torch.cuda.manual_seed_all(seed)


seed_everything(CFG.SEED)

print(f"Device: {CFG.DEVICE}")

In [ ]:
# ====================================================
# COCO -> YOLO-seg conversion
# ====================================================

with open(CFG.TRAIN_JSON, "r") as f:
    coco = json.load(f)

images = {img["id"]: img for img in coco["images"]}

annotations = {}

for ann in coco["annotations"]:
    annotations.setdefault(ann["image_id"], []).append(ann)

file_to_ids = {}

for image_id, info in images.items():
    file_to_ids.setdefault(info["file_name"], []).append(image_id)

unique_files = list(file_to_ids.keys())

def observation_completeness(image_id):
    anns = annotations.get(image_id, [])
    total_area = sum(ann.get("area", 0) for ann in anns)
    return (len(anns), total_area)

file_to_best_id = {
    file_name: max(ids, key=observation_completeness)
    for file_name, ids in file_to_ids.items()
}

n_duplicated = sum(1 for ids in file_to_ids.values() if len(ids) > 1)
print(f"Instances with multiple labelers: "
      f"{n_duplicated} / {len(file_to_ids)}")

unique_files = list(file_to_best_id.keys())

train_files, valid_files = train_test_split(
    unique_files,
    test_size=0.2,
    random_state=CFG.SEED
)

train_ids = [file_to_best_id[f] for f in train_files]
valid_ids = [file_to_best_id[f] for f in valid_files]

print(f"Train images: {len(train_ids)}")
print(f"Valid images: {len(valid_ids)}")


if CFG.USE_CATEGORIES and "categories" in coco and coco["categories"]:

    categories = sorted(coco["categories"], key=lambda c: c["id"])
    cat_id_to_yolo = {c["id"]: i for i, c in enumerate(categories)}
    class_names = [c["name"] for c in categories]

else:

    cat_id_to_yolo = None
    class_names = ["filament"]

print("Classes:", class_names)


def coco_polygon_to_yolo_line(polygon, w, h, cls_id):

    pts = np.array(polygon, dtype=np.float32).reshape(-1, 2)

    pts[:, 0] = np.clip(pts[:, 0] / w, 0, 1)
    pts[:, 1] = np.clip(pts[:, 1] / h, 0, 1)

    coords = " ".join(f"{v:.6f}" for v in pts.flatten())

    return f"{cls_id} {coords}"


def build_yolo_split(split_name, image_ids):

    img_dir = os.path.join(CFG.YOLO_DIR, "images", split_name)
    lbl_dir = os.path.join(CFG.YOLO_DIR, "labels", split_name)

    os.makedirs(img_dir, exist_ok=True)
    os.makedirs(lbl_dir, exist_ok=True)

    n_skipped = 0

    for image_id in image_ids:

        info = images[image_id]
        h, w = info["height"], info["width"]
        stem = os.path.splitext(info["file_name"])[0]

        src = os.path.join(CFG.TRAIN_DIR, info["file_name"])
        dst = os.path.join(img_dir, info["file_name"])

        if not os.path.exists(dst):
            os.symlink(src, dst)

        lines = []

        for ann in annotations.get(image_id, []):

            if ann.get("iscrowd", 0):
                n_skipped += 1
                continue

            if cat_id_to_yolo is not None:
                cls_id = cat_id_to_yolo[ann["category_id"]]
            else:
                cls_id = 0

            for polygon in ann["segmentation"]:

                if len(polygon) < 6:
                    n_skipped += 1
                    continue

                lines.append(
                    coco_polygon_to_yolo_line(polygon, w, h, cls_id)
                )

        with open(os.path.join(lbl_dir, stem + ".txt"), "w") as f:
            f.write("\n".join(lines))

    return n_skipped


n_skip_train = build_yolo_split("train", train_ids)
n_skip_valid = build_yolo_split("val", valid_ids)

print(f"Skipped (iscrowd/degenerate): train={n_skip_train}, val={n_skip_valid}")

names_yaml = "\n".join(f"  {i}: {name}" for i, name in enumerate(class_names))

data_yaml = f"""path: {CFG.YOLO_DIR}
train: images/train
val: images/val
names:
{names_yaml}
"""

with open(os.path.join(CFG.YOLO_DIR, "data.yaml"), "w") as f:
    f.write(data_yaml)

print("YOLO dataset ready:", CFG.YOLO_DIR)

In [ ]:
_n_gpu = torch.cuda.device_count()
if _n_gpu > 1:
    CFG.DEVICE = ",".join(str(i) for i in range(_n_gpu))
elif _n_gpu == 1:
    CFG.DEVICE = 0
else:
    CFG.DEVICE = "cpu"

In [ ]:
# ====================================================
# Train
# ====================================================

model = YOLO(CFG.MODEL)

model.train(
    data=os.path.join(CFG.YOLO_DIR, "data.yaml"),
    epochs=CFG.EPOCHS,
    imgsz=CFG.IMGSZ,
    batch=CFG.BATCH_SIZE,
    device=CFG.DEVICE,
    patience=CFG.PATIENCE,
    seed=CFG.SEED,
    single_cls=(not CFG.USE_CATEGORIES),
    mask_ratio=CFG.MASK_RATIO,
    overlap_mask=False,
    hsv_h=0.0,
    hsv_s=0.0,
    fliplr=0.0,
    flipud=0.0,

    mosaic=0.3,
    degrees=10,
    scale=0.3,
    hsv_v=0.3,
    close_mosaic=5,

    project="runs_filament",
    name="yolo_seg_aug_test",
)

import shutil

best_weights_path = str(model.trainer.best)
shutil.copy(best_weights_path, "best_model.pt")

print("Best weights:", best_weights_path, "-> saved as best_model.pt")

In [ ]:
# ============================================================
# Crop-refiner: light UNet over instances, found by YOLO
# ============================================================
import torch
import torch.nn as nn
from torch.utils.data import Dataset, DataLoader
from torchvision.models import resnet18, ResNet18_Weights
import numpy as np
import cv2
import random
import os
from tqdm.auto import tqdm

CROP_SIZE = 256
CONTEXT = 1.8
MIN_CROP = 96


def square_bounds(mask, context=CONTEXT, minimum=MIN_CROP):
    height, width = mask.shape
    ys, xs = np.where(mask)
    if len(xs) == 0:
        return 0, 0, width, height
    cx, cy = (xs.min() + xs.max() + 1) / 2, (ys.min() + ys.max() + 1) / 2
    side = int(np.ceil(max(xs.max() - xs.min() + 1, ys.max() - ys.min() + 1) * context))
    side = min(max(side, minimum), min(height, width))
    x0, y0 = int(round(cx - side / 2)), int(round(cy - side / 2))
    x1, y1 = x0 + side, y0 + side
    if x0 < 0: x1 -= x0; x0 = 0
    if y0 < 0: y1 -= y0; y0 = 0
    if x1 > width: x0 -= x1 - width; x1 = width
    if y1 > height: y0 -= y1 - height; y1 = height
    return int(x0), int(y0), int(x1), int(y1)


class ConvBlock(nn.Sequential):
    def __init__(self, in_ch, out_ch):
        super().__init__(
            nn.Conv2d(in_ch, out_ch, 3, padding=1, bias=False), nn.BatchNorm2d(out_ch), nn.ReLU(inplace=True),
            nn.Conv2d(out_ch, out_ch, 3, padding=1, bias=False), nn.BatchNorm2d(out_ch), nn.ReLU(inplace=True),
        )


class CropUNet(nn.Module):

    def __init__(self, pretrained=True):
        super().__init__()
        enc = resnet18(weights=ResNet18_Weights.DEFAULT if pretrained else None)
        self.stem = nn.Sequential(enc.conv1, enc.bn1, enc.relu)
        self.pool, self.layer1, self.layer2, self.layer3, self.layer4 = (
            enc.maxpool, enc.layer1, enc.layer2, enc.layer3, enc.layer4
        )
        self.up4, self.dec4 = nn.ConvTranspose2d(512, 256, 2, 2), ConvBlock(512, 256)
        self.up3, self.dec3 = nn.ConvTranspose2d(256, 128, 2, 2), ConvBlock(256, 128)
        self.up2, self.dec2 = nn.ConvTranspose2d(128, 64, 2, 2), ConvBlock(128, 64)
        self.up1, self.dec1 = nn.ConvTranspose2d(64, 32, 2, 2), ConvBlock(96, 32)
        self.up0, self.head = nn.ConvTranspose2d(32, 16, 2, 2), nn.Conv2d(16, 1, 1)
        self.register_buffer("mean", torch.tensor([0.485, 0.456, 0.406]).view(1, 3, 1, 1))
        self.register_buffer("std", torch.tensor([0.229, 0.224, 0.225]).view(1, 3, 1, 1))

    def forward(self, image):
        image = (image - self.mean) / self.std
        x0 = self.stem(image)
        x1 = self.layer1(self.pool(x0))
        x2 = self.layer2(x1)
        x3 = self.layer3(x2)
        x4 = self.layer4(x3)
        x = self.dec4(torch.cat([self.up4(x4), x3], 1))
        x = self.dec3(torch.cat([self.up3(x), x2], 1))
        x = self.dec2(torch.cat([self.up2(x), x1], 1))
        x = self.dec1(torch.cat([self.up1(x), x0], 1))
        return self.head(self.up0(x))

In [ ]:
# ============================================================
# Crop cash from GT-labels (train_ids/valid_ids from YOLO-conversion)
# ============================================================

def build_crop_arrays(image_ids):
    imgs, masks = [], []
    for image_id in tqdm(image_ids, desc="building crops"):
        info = images[image_id]
        image = cv2.imread(os.path.join(CFG.TRAIN_DIR, info["file_name"]), cv2.IMREAD_GRAYSCALE)
        for ann in annotations.get(image_id, []):
            m = np.zeros(image.shape, dtype=np.uint8)
            for polygon in ann["segmentation"]:
                pts = np.round(np.array(polygon, dtype=np.float32).reshape(-1, 2)).astype(np.int32)
                cv2.fillPoly(m, [pts], 1)
            if m.sum() == 0:
                continue
            x0, y0, x1, y1 = square_bounds(m)
            imgs.append(cv2.resize(image[y0:y1, x0:x1], (CROP_SIZE, CROP_SIZE), interpolation=cv2.INTER_LINEAR))
            masks.append(cv2.resize(m[y0:y1, x0:x1], (CROP_SIZE, CROP_SIZE), interpolation=cv2.INTER_NEAREST))
    return np.stack(imgs), np.stack(masks)


train_crop_images, train_crop_masks = build_crop_arrays(train_ids)
valid_crop_images, valid_crop_masks = build_crop_arrays(valid_ids)
print(f"train crops: {len(train_crop_images)}, valid crops: {len(valid_crop_images)}")


class CropDataset(Dataset):
    def __init__(self, imgs, masks, augment=False):
        self.imgs, self.masks, self.augment = imgs, masks, augment

    def __len__(self):
        return len(self.imgs)

    def __getitem__(self, idx):
        image = torch.from_numpy(self.imgs[idx].copy()).float() / 255
        mask = torch.from_numpy(self.masks[idx].copy()).float().unsqueeze(0)

        if self.augment:
            k = random.randrange(4)
            image, mask = torch.rot90(image, k, (0, 1)), torch.rot90(mask, k, (1, 2))
            if random.random() < 0.5:
                image, mask = torch.flip(image, (1,)), torch.flip(mask, (2,))
            if random.random() < 0.5:
                image = image ** random.uniform(0.85, 1.20)  # gamma jitter
            if random.random() < 0.5:
                image = torch.clamp(image * random.uniform(0.90, 1.10), 0, 1)  # gain jitter

        return image.unsqueeze(0).repeat(3, 1, 1), mask


train_crop_loader = DataLoader(CropDataset(train_crop_images, train_crop_masks, augment=True),
                                batch_size=16, shuffle=True, num_workers=2, pin_memory=True)
valid_crop_loader = DataLoader(CropDataset(valid_crop_images, valid_crop_masks, augment=False),
                                batch_size=32, shuffle=False, num_workers=2, pin_memory=True)

In [ ]:
# ============================================================
# Train
# ============================================================

def soft_dice_loss(logits, targets):
    probs = logits.sigmoid()
    inter = (probs * targets).sum((1, 2, 3))
    denom = probs.sum((1, 2, 3)) + targets.sum((1, 2, 3))
    return (1 - (2 * inter + 1) / (denom + 1)).mean()


def train_refiner(model, train_loader, valid_loader, epochs, out_path):
    device = torch.device("cuda:0" if torch.cuda.is_available() else "cpu")
    model.to(device)
    optimizer = torch.optim.AdamW(model.parameters(), lr=3e-4, weight_decay=1e-4)
    scheduler = torch.optim.lr_scheduler.CosineAnnealingLR(optimizer, T_max=epochs)
    scaler = torch.amp.GradScaler("cuda", enabled=torch.cuda.is_available())
    best_dice = -1

    for epoch in range(1, epochs + 1):
        model.train()
        for imgs, masks in tqdm(train_loader, desc=f"refiner epoch {epoch}/{epochs}", leave=False):
            imgs, masks = imgs.to(device), masks.to(device)
            optimizer.zero_grad(set_to_none=True)
            with torch.amp.autocast("cuda", enabled=torch.cuda.is_available()):
                logits = model(imgs)
                loss = 0.5 * nn.functional.binary_cross_entropy_with_logits(logits, masks) + 0.5 * soft_dice_loss(logits, masks)
            scaler.scale(loss).backward()
            scaler.step(optimizer)
            scaler.update()

        model.eval()
        dice_sum = n = 0
        with torch.no_grad():
            for imgs, masks in valid_loader:
                imgs, masks = imgs.to(device), masks.to(device).bool()
                pred = model(imgs).sigmoid() >= 0.65
                inter = (pred & masks).sum((1, 2, 3)).float()
                denom = pred.sum((1, 2, 3)) + masks.sum((1, 2, 3))
                dice_sum += ((2 * inter + 1e-7) / (denom + 1e-7)).sum().item()
                n += len(imgs)
        val_dice = dice_sum / n
        print(f"epoch {epoch}: val_dice={val_dice:.4f}")

        if val_dice > best_dice:
            best_dice = val_dice
            torch.save(model.state_dict(), out_path)

        scheduler.step()

    return best_dice


refiner = CropUNet(pretrained=True)
best_refiner_dice = train_refiner(refiner, train_crop_loader, valid_crop_loader, epochs=8, out_path="best_refiner.pt")

In [ ]:
# ============================================================
# Inference: YOLO finds instacnes -> refiner adds details
# ============================================================

device = torch.device("cuda:0" if torch.cuda.is_available() else "cpu")
refiner = CropUNet(pretrained=False)
refiner.load_state_dict(torch.load("best_refiner.pt", map_location=device))
refiner.to(device).eval()
REFINER_THRESHOLD = 0.65


@torch.no_grad()
def refine_masks(image_gray, yolo_masks, refiner, device, threshold=REFINER_THRESHOLD):
    refined = []
    for mask in yolo_masks:
        if mask.sum() == 0:
            refined.append(mask)
            continue
        x0, y0, x1, y1 = square_bounds(mask)
        crop = cv2.resize(image_gray[y0:y1, x0:x1], (CROP_SIZE, CROP_SIZE), interpolation=cv2.INTER_LINEAR)
        tensor = torch.from_numpy(crop).float().unsqueeze(0).unsqueeze(0).repeat(1, 3, 1, 1).to(device) / 255
        prob = refiner(tensor).sigmoid()[0, 0].cpu().numpy()
        crop_mask = cv2.resize(prob, (x1 - x0, y1 - y0), interpolation=cv2.INTER_LINEAR) >= threshold

        if crop_mask.any():
            full = np.zeros_like(mask)
            full[y0:y1, x0:x1] = crop_mask
            refined.append(full)
        else:
            refined.append(mask)

    return refined

In [ ]:
import cv2

def build_full_mask(image_id, images, annotations):

    info = images[image_id]
    h, w = info["height"], info["width"]

    mask = np.zeros((h, w), dtype=np.uint8)

    for ann in annotations.get(image_id, []):

        for polygon in ann["segmentation"]:

            pts = np.array(polygon, dtype=np.float32).reshape(-1, 2)
            pts = np.round(pts).astype(np.int32)

            cv2.fillPoly(mask, [pts], 1)

    return mask

In [ ]:
import matplotlib.pyplot as plt
import cv2
import numpy as np
import random

best_model = YOLO("best_model.pt")

N_SAMPLES = 4
sample_ids = random.sample(list(valid_ids), min(N_SAMPLES, len(valid_ids)))

fig, axes = plt.subplots(2, N_SAMPLES, figsize=(6 * N_SAMPLES, 11))


def overlay_vs_gt(img_rgb, gt_mask, pred_masks, min_size):

    pred_union = np.zeros_like(gt_mask)
    for m in pred_masks:
        if m.sum() >= min_size:
            pred_union = np.maximum(pred_union, m)

    gt_bool, pred_bool = gt_mask.astype(bool), pred_union.astype(bool)

    overlay = img_rgb.copy()
    overlay[gt_bool & pred_bool] = [0, 255, 0]
    overlay[gt_bool & ~pred_bool] = [255, 255, 0]
    overlay[~gt_bool & pred_bool] = [255, 0, 0] 

    return cv2.addWeighted(img_rgb, 0.4, overlay, 0.6, 0), len(pred_masks)


for col, image_id in enumerate(sample_ids):

    info = images[image_id]
    image_path = os.path.join(CFG.TRAIN_DIR, info["file_name"])

    img_gray = cv2.imread(image_path, cv2.IMREAD_GRAYSCALE)
    img_rgb = cv2.cvtColor(img_gray, cv2.COLOR_GRAY2RGB)

    gt_mask = build_full_mask(image_id, images, annotations)

    r = best_model.predict(
        image_path, imgsz=CFG.IMGSZ, conf=CFG.CONF_THRESHOLD,
        agnostic_nms=True, retina_masks=True, verbose=False
    )[0]

    yolo_masks = [m.cpu().numpy().astype(np.uint8) for m in r.masks.data] if r.masks is not None else []
    refined_masks = refine_masks(img_gray, yolo_masks, refiner, device) if yolo_masks else []

    # --- строка 1: YOLO without refiner ---
    blended_yolo, n_yolo = overlay_vs_gt(img_rgb, gt_mask, yolo_masks, CFG.MIN_COMPONENT_SIZE)
    axes[0, col].imshow(blended_yolo)
    axes[0, col].set_title(f"{info['file_name']}\nYOLO only | GT:{len(cv2.findContours(gt_mask, cv2.RETR_EXTERNAL, cv2.CHAIN_APPROX_SIMPLE)[0])} Pred:{n_yolo}", fontsize=9)
    axes[0, col].axis("off")

    # --- строка 2: YOLO + refiner ---
    blended_refined, n_refined = overlay_vs_gt(img_rgb, gt_mask, refined_masks, CFG.MIN_COMPONENT_SIZE)
    axes[1, col].imshow(blended_refined)
    axes[1, col].set_title(f"+ refiner | Pred:{n_refined}", fontsize=9)
    axes[1, col].axis("off")

plt.tight_layout()
plt.savefig("visual_audit_refiner.png", dpi=120)
plt.show()

In [ ]:
import pandas as pd
from pycocotools import mask as mask_utils

refiner = CropUNet(pretrained=False)
refiner.load_state_dict(torch.load("best_refiner.pt", map_location=device))
refiner.to(device).eval()

test_files = sorted([
    f for f in os.listdir(CFG.TEST_DIR)
    if f.endswith(".jpeg")
])

rows = []

for filename in tqdm(test_files, desc="YOLO + refiner inference"):

    image_path = os.path.join(CFG.TEST_DIR, filename)
    image_gray = cv2.imread(image_path, cv2.IMREAD_GRAYSCALE)

    r = best_model.predict(
        image_path,
        imgsz=CFG.IMGSZ,
        conf=CFG.CONF_THRESHOLD,
        agnostic_nms=True,
        retina_masks=True,
        verbose=False
    )[0]

    if r.masks is None:
        continue

    yolo_masks = [m.cpu().numpy().astype(np.uint8) for m in r.masks.data]

    refined = refine_masks(image_gray, yolo_masks, refiner, device)

    image_id = filename.replace(".jpeg", "")
    counter = 1

    for mask in refined:

        if mask.sum() < CFG.MIN_COMPONENT_SIZE:
            continue

        rle = mask_utils.encode(np.asfortranarray(mask.astype(np.uint8)))

        rows.append({
            "filament_id": f"{image_id}_{counter}",
            "segmentation_rle": rle["counts"].decode("utf-8")
        })

        counter += 1

submission = pd.DataFrame(rows)
OUTPUT_PATH = "/kaggle/working/submission_v26_realdata_yolo_refiner.csv"
submission.to_csv(OUTPUT_PATH, index=False, lineterminator="\n")

print(submission.head())
print(f"Total predicted filaments: {len(submission)}")
print("Saved:", OUTPUT_PATH)

In [ ]:
# Final structural validation with the official pycocotools decoder.
from pathlib import Path
checked = pd.read_csv(OUTPUT_PATH, keep_default_na=False)
expected_columns = ['filament_id', 'segmentation_rle']
assert checked.columns.tolist() == expected_columns
assert len(checked) > 0
assert not checked.isna().any().any()
assert not checked['filament_id'].duplicated().any()
assert not checked['segmentation_rle'].str.strip().eq('').any()
test_ids = {Path(name).stem for name in test_files}
submitted_ids = {value.rsplit('_', 1)[0] for value in checked['filament_id']}
assert submitted_ids <= test_ids
areas = []
for counts in tqdm(checked['segmentation_rle'], desc='official RLE validation'):
    decoded = mask_utils.decode({
        'size': [2048, 2048],
        'counts': counts.encode('utf-8'),
    })
    assert decoded.shape == (2048, 2048)
    area = int(decoded.sum())
    assert area > 0
    areas.append(area)
print({
    'validated_csv': OUTPUT_PATH,
    'rows': len(checked),
    'images_with_predictions': len(submitted_ids),
    'test_images': len(test_ids),
    'area_min': min(areas),
    'area_median': int(np.median(areas)),
    'area_max': max(areas),
})